In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/insurance_data.csv')

# Calculate derived metrics
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']
df['LossRatio'] = df['LossRatio'].replace([np.inf, -np.inf], np.nan)

# Margin = TotalPremium - TotalClaims
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

# Claim Frequency (1 if claim > 0, else 0)
df['ClaimFrequency'] = (df['TotalClaims'] > 0).astype(int)

# Claim Severity (only where claim > 0)
df['ClaimSeverity'] = df['TotalClaims'].where(df['TotalClaims'] > 0, np.nan)

print("Data loaded successfully")
print(f"Shape: {df.shape}")
print("\nCalculated metrics:")
print(f"Loss Ratio - mean: {df['LossRatio'].mean():.4f}")
print(f"Margin - mean: {df['Margin'].mean():.2f}")
print(f"Claim Frequency - mean: {df['ClaimFrequency'].mean():.4f}")

Data loaded successfully
Shape: (10000, 25)

Calculated metrics:
Loss Ratio - mean: 0.4428
Margin - mean: 1173.94
Claim Frequency - mean: 0.1535


In [2]:
print("=" * 60)
print("HYPOTHESIS 1: Risk differences across provinces")
print("=" * 60)

# Group Loss Ratio by Province
province_groups = [group['LossRatio'].dropna().values for name, group in df.groupby('Province') if len(group) > 10]

# ANOVA test
if len(province_groups) >= 2:
    f_stat, p_value = f_oneway(*province_groups)
    print(f"ANOVA F-statistic: {f_stat:.4f}")
    print(f"P-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print("\n✅ REJECT H₀: There ARE significant risk differences across provinces")
        print("Business recommendation: Adjust premiums based on province risk profile")
    else:
        print("\n❌ FAIL TO REJECT H₀: No significant risk differences across provinces")
else:
    print("Insufficient data for ANOVA test")

# Show province loss ratios
province_loss = df.groupby('Province')['LossRatio'].mean().sort_values(ascending=False)
print("\nLoss Ratio by Province:")
print(province_loss)

HYPOTHESIS 1: Risk differences across provinces
ANOVA F-statistic: 1.9925
P-value: 0.092767

❌ FAIL TO REJECT H₀: No significant risk differences across provinces

Loss Ratio by Province:
Province
Somali         0.523516
Oromia         0.452680
Tigray         0.438639
Addis Ababa    0.436347
Amhara         0.395941
Name: LossRatio, dtype: float64


In [4]:
print(df.columns.tolist())

['CustomerID', 'Age', 'Gender', 'Province', 'VehicleType', 'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD', 'PastClaims', 'Claimed', 'ClaimAmount', 'TotalPremium', 'TotalClaims', 'CoverType', 'AutoMake', 'VehicleModel', 'CustomValueEstimate', 'ZipCode', 'TransactionDate', 'LossRatio', 'Margin', 'ClaimFrequency', 'ClaimSeverity']


In [5]:
print("=" * 60)
print("HYPOTHESIS 2: Risk differences between zip codes")
print("=" * 60)

# Group Claim Frequency by ZipCode
zipcode_groups = [group['ClaimFrequency'].dropna().values for name, group in df.groupby('ZipCode') if len(group) > 10]

if len(zipcode_groups) >= 2:
    f_stat, p_value_h2 = f_oneway(*zipcode_groups)
    print(f"ANOVA F-statistic: {f_stat:.4f}")
    print(f"P-value: {p_value_h2:.6f}")
    
    if p_value_h2 < 0.05:
        print("\n✅ REJECT H₀: There ARE significant risk differences between zip codes")
        print("Business recommendation: Use zip code as a pricing factor")
    else:
        print("\n❌ FAIL TO REJECT H₀: No significant risk differences between zip codes")
else:
    print("Insufficient data for ANOVA test")
    p_value_h2 = 1.0

# Show top zip codes by claim frequency
top_zipcodes = df.groupby('ZipCode')['ClaimFrequency'].mean().sort_values(ascending=False).head(10)
print("\nTop 10 Zip Codes by Claim Frequency:")
print(top_zipcodes)

HYPOTHESIS 2: Risk differences between zip codes
ANOVA F-statistic: 1.7235
P-value: 0.015343

✅ REJECT H₀: There ARE significant risk differences between zip codes
Business recommendation: Use zip code as a pricing factor

Top 10 Zip Codes by Claim Frequency:
ZipCode
40002    0.205323
40005    0.202073
40004    0.193277
30001    0.176923
10002    0.173497
20003    0.170478
10001    0.170423
50004    0.168831
20002    0.167015
20005    0.161538
Name: ClaimFrequency, dtype: float64


In [6]:
print("=" * 60)
print("HYPOTHESIS 3: Margin differences between zip codes")
print("=" * 60)

# Group Margin by ZipCode
margin_groups = [group['Margin'].dropna().values for name, group in df.groupby('ZipCode') if len(group) > 10]

if len(margin_groups) >= 2:
    f_stat, p_value_h3 = f_oneway(*margin_groups)
    print(f"ANOVA F-statistic: {f_stat:.4f}")
    print(f"P-value: {p_value_h3:.6f}")
    
    if p_value_h3 < 0.05:
        print("\n✅ REJECT H₀: There ARE significant margin differences between zip codes")
        print("Business recommendation: Target high-margin zip codes for marketing")
    else:
        print("\n❌ FAIL TO REJECT H₀: No significant margin differences between zip codes")
else:
    print("Insufficient data for ANOVA test")
    p_value_h3 = 1.0

# Show top zip codes by margin
top_margin = df.groupby('ZipCode')['Margin'].mean().sort_values(ascending=False).head(10)
print("\nTop 10 Zip Codes by Average Margin:")
print(top_margin)

HYPOTHESIS 3: Margin differences between zip codes
ANOVA F-statistic: 1.3041
P-value: 0.145766

❌ FAIL TO REJECT H₀: No significant margin differences between zip codes

Top 10 Zip Codes by Average Margin:
ZipCode
30002    1624.901478
10003    1526.428571
20001    1429.907757
50002    1396.013423
50001    1376.013158
40003    1307.230469
10004    1298.916780
30003    1257.281863
30004    1242.167539
50003    1236.128655
Name: Margin, dtype: float64


In [ ]:
p_value_h1 = p_value

In [8]:
p_value_h2 = p_value

In [9]:
p_value_h3 = p_value

In [10]:
p_value_h4 = p_value

In [11]:
# Create results summary
results = {
    'Hypothesis': [
        'H1: Provinces (Loss Ratio)',
        'H2: Zip Codes (Claim Frequency)',
        'H3: Zip Codes (Margin)',
        'H4: Gender (Claim Severity)'
    ],
    'Test': ['ANOVA', 'ANOVA', 'ANOVA', 'T-test'],
    'P-Value': [p_value_h1, p_value_h2, p_value_h3, p_value_h4],
    'Reject H₀ (p < 0.05)': [
        p_value_h1 < 0.05,
        p_value_h2 < 0.05,
        p_value_h3 < 0.05,
        p_value_h4 < 0.05
    ]
}

results_df = pd.DataFrame(results)
print("=" * 70)
print("HYPOTHESIS TESTING SUMMARY")
print("=" * 70)
print(results_df.to_string(index=False))

HYPOTHESIS TESTING SUMMARY
                     Hypothesis   Test  P-Value  Reject H₀ (p < 0.05)
     H1: Provinces (Loss Ratio)  ANOVA 0.092767                 False
H2: Zip Codes (Claim Frequency)  ANOVA 0.092767                 False
         H3: Zip Codes (Margin)  ANOVA 0.092767                 False
    H4: Gender (Claim Severity) T-test 0.092767                 False
